In [ ]:
import os, sys, glob, random, warnings, time, json, shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from PIL import Image
import cv2
from tqdm.notebook import tqdm
import scipy.stats as stats
from scipy import ndimage
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             average_precision_score, confusion_matrix)
from sklearn.calibration import calibration_curve
import statsmodels.api as sm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.cuda.amp import GradScaler, autocast

import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')

# Verify versions
print("="*60)
print("Library Version Check")
print("="*60)
for lib, mod in [("NumPy", np), ("Pandas", pd), ("PyTorch", torch),
                 ("OpenCV", cv2), ("Albumentations", A)]:
    print(f"  {lib}: {mod.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
print("="*60)

In [ ]:
CFG = {
    # ── Paths ──────────────────────────────────────────────
    "data_root"    : "/kaggle/input/datasets/thanhbnhphan/hc18-grand-challenge",
    "train_img_dir": "/kaggle/input/datasets/thanhbnhphan/hc18-grand-challenge/training_set",
    "test_img_dir" : "/kaggle/input/datasets/thanhbnhphan/hc18-grand-challenge/test_set",
    "train_csv"    : "/kaggle/input/datasets/thanhbnhphan/hc18-grand-challenge/training_set_pixel_size_and_HC.csv",
    "test_csv"     : "/kaggle/input/datasets/thanhbnhphan/hc18-grand-challenge/test_set_pixel_size.csv",
    "output_root"  : "/kaggle/working/outputs",

    # ── Reproducibility ────────────────────────────────────
    "seed": 42,

    # ── Image ──────────────────────────────────────────────
    "img_size"    : 512,
    "in_channels" : 1,
    "num_classes" : 1,

    # ── Training ───────────────────────────────────────────
    "batch_size"  : 8,
    "num_epochs"  : 100,
    "lr"          : 1e-4,
    "weight_decay": 1e-5,
    "val_split"   : 0.15,
    "patience"    : 15,
    "min_delta"   : 1e-4,

    # ── Augmentation ───────────────────────────────────────
    "aug_p"       : 0.5,

    # ── MC Dropout ─────────────────────────────────────────
    "mc_passes"   : 20,
    "dropout_rate": 0.3,

    # ── Reliability Score ──────────────────────────────────
    "rs_weights"  : {
        "confidence" : 0.25,
        "entropy"    : 0.20,
        "boundary"   : 0.20,
        "consistency": 0.20,
        "morphology" : 0.15,
    },

    # ── Failure Detection ──────────────────────────────────
    "dice_fail_threshold": 0.70,

    # ── Calibration ────────────────────────────────────────
    "n_bins": 15,

    # ── Publication figures ────────────────────────────────
    "dpi"    : 300,
    "figsize": (12, 8),
    "font_size": 12,
}

# Set random seeds everywhere
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(CFG["seed"])

# Create output directory tree
DIRS = {
    "root"         : CFG["output_root"],
    "audit"        : f"{CFG['output_root']}/01_data_audit",
    "eda"          : f"{CFG['output_root']}/02_eda",
    "preproc"      : f"{CFG['output_root']}/03_preprocessing",
    "models"       : f"{CFG['output_root']}/04_models",
    "unet"         : f"{CFG['output_root']}/04_models/unet",
    "attunet"      : f"{CFG['output_root']}/04_models/attention_unet",
    "unetpp"       : f"{CFG['output_root']}/04_models/unetpp",
    "nnunet"       : f"{CFG['output_root']}/04_models/nnunet",
    "uncertainty"  : f"{CFG['output_root']}/05_uncertainty",
    "reliability"  : f"{CFG['output_root']}/06_reliability",
    "failure"      : f"{CFG['output_root']}/07_failure_detection",
    "calibration"  : f"{CFG['output_root']}/08_calibration",
    "statistics"   : f"{CFG['output_root']}/09_statistics",
    "boundary"     : f"{CFG['output_root']}/10_boundary",
    "taxonomy"     : f"{CFG['output_root']}/11_taxonomy",
    "figures"      : f"{CFG['output_root']}/12_figures",
    "predictions"  : f"{CFG['output_root']}/13_predictions",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print("All output directories created.")
print(json.dumps({k: v for k, v in CFG.items() if k != "rs_weights"}, indent=2))


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from glob import glob
from tqdm import tqdm

DATA_DIR = "/kaggle/input/datasets/thanhbnhphan/hc18-grand-challenge/training_set"

all_files = glob(os.path.join(DATA_DIR, "*.png"))

image_paths = []
mask_paths = []

for file in all_files:
    filename = os.path.basename(file)

    if "_Annotation" in filename:
        mask_paths.append(file)
    else:
        image_paths.append(file)

image_paths = sorted(image_paths)
mask_paths = sorted(mask_paths)

print("Images:", len(image_paths))
print("Masks :", len(mask_paths))

In [ ]:
def discover_dataset(train_dir: str, test_dir: str) -> tuple:
    """Walk training and test directories; return tidy DataFrames."""

    # ── Training images & masks ────────────────────────────
    all_files  = sorted(glob.glob(os.path.join(train_dir, "*.png")))
    img_files  = [f for f in all_files if "_Annotation" not in f]
    mask_files = [f for f in all_files if "_Annotation"     in f]

    train_records = []
    for img_path in img_files:
        stem      = Path(img_path).stem          # e.g. "001_HC"
        mask_name = stem + "_Annotation.png"
        mask_path = os.path.join(train_dir, mask_name)
        train_records.append({
            "img_path"   : img_path,
            "mask_path"  : mask_path if os.path.exists(mask_path) else None,
            "has_mask"   : os.path.exists(mask_path),
            "split"      : "train",
            "sample_id"  : stem.replace("_HC", ""),
        })

    train_df = pd.DataFrame(train_records)

    # ── Test images ────────────────────────────────────────
    test_files = sorted(glob.glob(os.path.join(test_dir, "*.png")))
    test_records = [{
        "img_path" : f,
        "mask_path": None,
        "has_mask" : False,
        "split"    : "test",
        "sample_id": Path(f).stem.replace("_HC", ""),
    } for f in test_files]
    test_df = pd.DataFrame(test_records)

    # ── Save manifest ──────────────────────────────────────
    manifest = pd.concat([train_df, test_df], ignore_index=True)
    manifest.to_csv(f"{DIRS['audit']}/dataset_manifest.csv", index=False)

    return train_df, test_df, manifest

train_df, test_df, manifest_df = discover_dataset(
    CFG["train_img_dir"], CFG["test_img_dir"]
)

print("="*60)
print("Dataset Discovery Summary")
print("="*60)
print(f"  Training images : {len(train_df)}")
print(f"  Training masks  : {train_df['has_mask'].sum()}")
print(f"  Missing masks   : {(~train_df['has_mask']).sum()}")
print(f"  Test images     : {len(test_df)}")
print(f"  Total files     : {len(manifest_df)}")
print("Manifest saved to:", f"{DIRS['audit']}/dataset_manifest.csv")


In [ ]:
def verify_image(path: str) -> dict:
    """Try to open an image and return basic metadata or error info."""
    result = {
        "path"      : path,
        "readable"  : False,
        "width"     : None,
        "height"    : None,
        "mode"      : None,
        "file_size" : os.path.getsize(path) if os.path.exists(path) else 0,
        "error"     : None,
    }
    try:
        img = Image.open(path)
        img.verify()                    # detects truncation
        img = Image.open(path)          # reopen after verify
        result.update({
            "readable": True,
            "width"   : img.width,
            "height"  : img.height,
            "mode"    : img.mode,
        })
    except Exception as e:
        result["error"] = str(e)
    return result

print("Verifying training images and masks …")
verification_records = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    img_info  = verify_image(row["img_path"])
    img_info["file_type"] = "image"

    if row["has_mask"]:
        msk_info  = verify_image(row["mask_path"])
        msk_info["file_type"] = "mask"
        # Check size consistency
        size_ok = (img_info["width"]  == msk_info["width"] and
                   img_info["height"] == msk_info["height"])
        img_info["size_match"] = size_ok
        msk_info["size_match"] = size_ok
        verification_records.append(msk_info)
    else:
        img_info["size_match"] = None

    verification_records.append(img_info)

verif_df = pd.DataFrame(verification_records)
verif_df.to_csv(f"{DIRS['audit']}/verification_report.csv", index=False)

# Summary statistics
n_corrupted   = verif_df[~verif_df["readable"]].shape[0]
n_size_issues = verif_df[verif_df["size_match"] == False].shape[0]
n_missing     = (~train_df["has_mask"]).sum()

print("="*60)
print("Verification Report")
print("="*60)
print(f"  Corrupted files      : {n_corrupted}")
print(f"  Size mismatch pairs  : {n_size_issues}")
print(f"  Missing masks        : {n_missing}")
print(f"  All images verified  : {verif_df[verif_df['file_type']=='image']['readable'].all()}")
print("Saved to:", f"{DIRS['audit']}/verification_report.csv")


In [ ]:
def extract_image_stats(row) -> dict:
    """Extract pixel-level and mask-level statistics from one sample."""
    img  = np.array(Image.open(row["img_path"]).convert("L"), dtype=np.float32)
    rec  = {
        "sample_id"         : row["sample_id"],
        "img_path"          : row["img_path"],
        "width"             : img.shape[1],
        "height"            : img.shape[0],
        "pixel_min"         : float(img.min()),
        "pixel_max"         : float(img.max()),
        "pixel_mean"        : float(img.mean()),
        "pixel_std"         : float(img.std()),
        "pixel_median"      : float(np.median(img)),
        "total_pixels"      : int(img.size),
    }
    if row["has_mask"]:
        msk = np.array(Image.open(row["mask_path"]).convert("L"), dtype=np.float32)
        msk_bin = (msk > 127).astype(np.uint8)
        fg      = msk_bin.sum()
        rec.update({
            "mask_area_px"      : int(fg),
            "foreground_pct"    : float(fg / msk_bin.size * 100),
            "mask_pixel_min"    : float(msk.min()),
            "mask_pixel_max"    : float(msk.max()),
        })
    return rec

print("Extracting per-image statistics …")
stats_records = [extract_image_stats(row)
                 for _, row in tqdm(train_df.iterrows(), total=len(train_df))]

stats_df = pd.DataFrame(stats_records)
stats_df.to_csv(f"{DIRS['audit']}/image_stats.csv", index=False)
print("Saved image_stats.csv  —  shape:", stats_df.shape)
print(stats_df.describe().T.round(3).to_string())

In [ ]:
summary_lines = [
    "=" * 60,
    "HC18 Fetal Ultrasound — Dataset Audit Report",
    "=" * 60,
    f"Training samples        : {len(train_df)}",
    f"Test samples            : {len(test_df)}",
    f"Samples with masks      : {train_df['has_mask'].sum()}",
    f"Missing masks           : {(~train_df['has_mask']).sum()}",
    f"Corrupted files         : {n_corrupted}",
    f"Size-mismatch pairs     : {n_size_issues}",
    "",
    "Image Dimensions",
    f"  Unique widths         : {stats_df['width'].unique().tolist()}",
    f"  Unique heights        : {stats_df['height'].unique().tolist()}",
    "",
    "Pixel Intensity (grayscale)",
    f"  Global mean           : {stats_df['pixel_mean'].mean():.2f}",
    f"  Global std            : {stats_df['pixel_std'].mean():.2f}",
    f"  Min across dataset    : {stats_df['pixel_min'].min():.2f}",
    f"  Max across dataset    : {stats_df['pixel_max'].max():.2f}",
    "",
    "Mask / Foreground",
    f"  Mean mask area (px)   : {stats_df['mask_area_px'].mean():.0f}",
    f"  Mean foreground %     : {stats_df['foreground_pct'].mean():.2f}%",
    f"  Min foreground %      : {stats_df['foreground_pct'].min():.2f}%",
    f"  Max foreground %      : {stats_df['foreground_pct'].max():.2f}%",
    "=" * 60,
]
report_text = "\n".join(summary_lines)
print(report_text)

with open(f"{DIRS['audit']}/dataset_summary_report.txt", "w") as f:
    f.write(report_text)

# Also save a machine-readable version
summary_dict = {
    "n_train": len(train_df),
    "n_test" : len(test_df),
    "n_with_mask": int(train_df["has_mask"].sum()),
    "n_missing_mask": int((~train_df["has_mask"]).sum()),
    "n_corrupted": n_corrupted,
    "n_size_mismatch": n_size_issues,
    "mean_pixel_mean": float(stats_df["pixel_mean"].mean()),
    "mean_fg_pct": float(stats_df["foreground_pct"].mean()),
}
pd.DataFrame([summary_dict]).to_csv(f"{DIRS['audit']}/dataset_report.csv", index=False)
print("Report saved.")


In [ ]:
plt.rcParams.update({
    "font.size"      : CFG["font_size"],
    "figure.dpi"     : 100,
    "savefig.dpi"    : CFG["dpi"],
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def save_fig(name: str, subdir: str = "eda"):
    path = f"{DIRS[subdir]}/{name}.png"
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    return path

# ── Figure A: Image size histogram ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(stats_df["width"],  bins=20, color="#4C72B0", edgecolor="white")
axes[0].set(title="Image Width Distribution",  xlabel="Width (px)",  ylabel="Count")
axes[1].hist(stats_df["height"], bins=20, color="#DD8452", edgecolor="white")
axes[1].set(title="Image Height Distribution", xlabel="Height (px)", ylabel="Count")
plt.suptitle("HC18 — Image Size Distributions", fontweight="bold")
save_fig("A_image_size_histogram")

# ── Figure B: Mask area histogram ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(stats_df["mask_area_px"].dropna(),    bins=30,
             color="#55A868", edgecolor="white")
axes[0].set(title="Mask Area (pixels)", xlabel="Pixels", ylabel="Count")
axes[1].hist(stats_df["foreground_pct"].dropna(),  bins=30,
             color="#C44E52", edgecolor="white")
axes[1].set(title="Foreground %", xlabel="%", ylabel="Count")
plt.suptitle("HC18 — Mask / Foreground Distributions", fontweight="bold")
save_fig("B_mask_area_histogram")

# ── Figure C: Pixel intensity histogram ───────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col, colour, title in zip(
        axes.flat,
        ["pixel_mean", "pixel_std", "pixel_min", "pixel_max"],
        ["#4C72B0",    "#DD8452",   "#55A868",   "#C44E52"],
        ["Mean Intensity", "Std Intensity", "Min Intensity", "Max Intensity"]):
    ax.hist(stats_df[col], bins=30, color=colour, edgecolor="white")
    ax.set(title=title, xlabel="Value", ylabel="Count")
plt.suptitle("HC18 — Pixel Intensity Distributions", fontweight="bold")
save_fig("C_pixel_intensity_histograms")

# ── Figure D: Correlation plot ────────────────────────────
corr_cols = ["pixel_mean", "pixel_std", "mask_area_px", "foreground_pct"]
corr_data = stats_df[corr_cols].dropna()
fig, ax   = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_data.corr(), annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=ax,
            square=True, linewidths=0.5)
ax.set_title("HC18 — Feature Correlation Matrix", fontweight="bold")
save_fig("D_correlation_heatmap")

# ── Figure E: Foreground % vs mask area scatter ────────────
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(stats_df["mask_area_px"], stats_df["foreground_pct"],
                alpha=0.6, c=stats_df["pixel_mean"], cmap="viridis", s=30)
plt.colorbar(sc, ax=ax, label="Mean Intensity")
ax.set(title="Mask Area vs Foreground %",
       xlabel="Mask Area (px)", ylabel="Foreground (%)")
save_fig("E_foreground_scatter")

print("EDA figures saved to:", DIRS["eda"])

# Export EDA results
eda_summary = {
    "same_size"          : stats_df[["width","height"]].nunique().max() == 1,
    "class_imbalance"    : stats_df["foreground_pct"].mean() < 10,
    "mean_fg_pct"        : float(stats_df["foreground_pct"].mean()),
    "mean_mask_area_px"  : float(stats_df["mask_area_px"].mean()),
}
pd.DataFrame([eda_summary]).to_csv(f"{DIRS['eda']}/eda_summary.csv", index=False)

print("\n=== EDA Conclusions ===")
print(f"  All images same size?      {eda_summary['same_size']}")
print(f"  Class imbalance (<10% fg)? {eda_summary['class_imbalance']}")
print(f"  Mean foreground %:         {eda_summary['mean_fg_pct']:.2f}%")
print(f"  Mean mask area (px):       {eda_summary['mean_mask_area_px']:.0f}")



In [ ]:
def zscore_normalise(img: np.ndarray) -> np.ndarray:
    """Zero-mean unit-variance normalisation per image."""
    mu, sigma = img.mean(), img.std()
    return (img - mu) / (sigma + 1e-8)

def minmax_normalise(img: np.ndarray) -> np.ndarray:
    """Min-max normalisation to [0, 1]."""
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-8)

def apply_clahe(img: np.ndarray, clip_limit: float = 2.0,
                tile_grid: tuple = (8, 8)) -> np.ndarray:
    """
    CLAHE — Contrast Limited Adaptive Histogram Equalization.
    Enhances local contrast in ultrasound images without amplifying noise.
    Works on uint8 images; returns uint8.
    """
    img_u8  = (np.clip(img, 0, 1) * 255).astype(np.uint8)
    clahe   = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(img_u8).astype(np.float32) / 255.0

def reduce_speckle(img: np.ndarray, d: int = 9,
                   sigma_color: float = 75,
                   sigma_space: float = 75) -> np.ndarray:
    """
    Bilateral filter for speckle noise reduction.
    Preserves edges while smoothing homogeneous regions.
    """
    img_u8 = (np.clip(img, 0, 1) * 255).astype(np.uint8)
    denoised = cv2.bilateralFilter(img_u8, d, sigma_color, sigma_space)
    return denoised.astype(np.float32) / 255.0

def median_filter(img: np.ndarray, ksize: int = 3) -> np.ndarray:
    """Median filter for salt-and-pepper noise."""
    img_u8 = (np.clip(img, 0, 1) * 255).astype(np.uint8)
    filtered = cv2.medianBlur(img_u8, ksize)
    return filtered.astype(np.float32) / 255.0

def preprocess_image(img_path: str,
                     target_size: int = 512,
                     use_clahe: bool = True,
                     use_speckle: bool = True) -> np.ndarray:
    """Full single-image preprocessing pipeline."""
    img = np.array(Image.open(img_path).convert("L"), dtype=np.float32)
    # Resize
    img = cv2.resize(img, (target_size, target_size),
                     interpolation=cv2.INTER_LINEAR)
    # Min-max to [0,1] before enhancement
    img = minmax_normalise(img)
    if use_speckle:
        img = reduce_speckle(img)
    if use_clahe:
        img = apply_clahe(img)
    # Z-score after enhancement
    img = zscore_normalise(img)
    return img

def preprocess_mask(mask_path: str, target_size: int = 512) -> np.ndarray:
    """Binary mask preprocessing."""
    msk = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)
    msk = cv2.resize(msk, (target_size, target_size),
                     interpolation=cv2.INTER_NEAREST)
    return (msk > 127).astype(np.float32)

print("Preprocessing functions defined.")

# ── Visualise preprocessing pipeline ──────────────────────
sample_row = train_df[train_df["has_mask"]].iloc[0]
raw   = np.array(Image.open(sample_row["img_path"]).convert("L"), dtype=np.float32)
raw   = cv2.resize(raw, (512, 512), interpolation=cv2.INTER_LINEAR)
raw_n = minmax_normalise(raw)

stages = {
    "Raw (resized)"  : raw_n,
    "After CLAHE"    : apply_clahe(raw_n),
    "After Bilateral": reduce_speckle(raw_n),
    "After Median"   : median_filter(raw_n),
    "Final (z-score)": preprocess_image(sample_row["img_path"]),
}

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
for ax, (title, img) in zip(axes, stages.items()):
    ax.imshow(img, cmap="gray")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.suptitle("Preprocessing Pipeline Stages", fontweight="bold", y=1.02)
save_fig("preprocessing_stages", "preproc")
print("Preprocessing visualisation saved.")


In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.ShiftScaleRotate(
        shift_limit=0.05, scale_limit=0.1,
        rotate_limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.5
    ),
    A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.3),
    A.GridDistortion(p=0.2),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),
    A.GaussianBlur(blur_limit=(3, 7), p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.2,
                               contrast_limit=0.2, p=0.4),
    A.Normalize(mean=0.0, std=1.0),
    ToTensorV2(),
], additional_targets={"mask": "mask"})

val_transform = A.Compose([
    A.Normalize(mean=0.0, std=1.0),
    ToTensorV2(),
], additional_targets={"mask": "mask"})

# Visualise augmentations
sample_img  = preprocess_image(sample_row["img_path"])
sample_mask = preprocess_mask(sample_row["mask_path"])
# Bring to [0,1] for augmentation display
sample_img_01 = minmax_normalise(sample_img)

fig, axes = plt.subplots(3, 6, figsize=(22, 11))
axes[0, 0].imshow(sample_img_01, cmap="gray"); axes[0, 0].set_title("Original Image")
axes[1, 0].imshow(sample_mask,   cmap="gray"); axes[1, 0].set_title("Original Mask")
axes[2, 0].imshow(
    np.ma.masked_where(sample_mask == 0, sample_mask),
    cmap="Reds", alpha=0.6); axes[2, 0].set_title("Overlay")
axes[2, 0].imshow(sample_img_01, cmap="gray", alpha=0.5)

aug_transform_vis = A.Compose([
    A.HorizontalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.ElasticTransform(alpha=120, sigma=6, p=0.8),
    A.GaussNoise(var_limit=(10, 30), p=0.5),
    A.RandomBrightnessContrast(p=0.5),
], additional_targets={"mask": "mask"})

seed_everything(CFG["seed"])
for col in range(1, 6):
    aug = aug_transform_vis(image=(sample_img_01 * 255).astype(np.uint8),
                            mask=(sample_mask * 255).astype(np.uint8))
    ai, am = aug["image"].astype(np.float32)/255, aug["mask"].astype(np.float32)/255
    axes[0, col].imshow(ai, cmap="gray"); axes[0, col].set_title(f"Aug {col}")
    axes[1, col].imshow(am, cmap="gray"); axes[1, col].set_title(f"Mask {col}")
    overlay = np.stack([ai]*3, axis=-1)
    axes[2, col].imshow(overlay)
    axes[2, col].imshow(am, cmap="Reds", alpha=0.4)
    axes[2, col].set_title(f"Overlay {col}")

for ax in axes.flat: ax.axis("off")
plt.suptitle("Augmentation Examples", fontweight="bold")
save_fig("augmentation_examples", "preproc")
print("Augmentation pipeline defined and visualised.")

In [ ]:
class HC18Dataset(Dataset):
    """PyTorch Dataset for HC18 fetal ultrasound segmentation."""

    def __init__(self, df: pd.DataFrame, transform=None,
                 target_size: int = 512, mode: str = "train"):
        self.df          = df.reset_index(drop=True)
        self.transform   = transform
        self.target_size = target_size
        self.mode        = mode  # "train" / "val" / "test"

    def __len__(self): return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]

        # Load and preprocess image
        img = preprocess_image(row["img_path"], self.target_size)
        # Bring to [0,1] for albumentations
        img_01 = minmax_normalise(img).astype(np.float32)

        if self.mode != "test" and row["has_mask"]:
            mask = preprocess_mask(row["mask_path"], self.target_size)
        else:
            mask = np.zeros((self.target_size, self.target_size), dtype=np.float32)

        if self.transform:
            aug  = self.transform(
                image=(img_01 * 255).astype(np.uint8),
                mask=(mask * 255).astype(np.uint8),
            )
            img_t  = aug["image"].float() / 255.0
            mask_t = aug["mask"].float()  / 255.0
        else:
            img_t  = torch.from_numpy(img_01).unsqueeze(0)
            mask_t = torch.from_numpy(mask)

        return {
            "image"    : img_t,
            "mask"     : mask_t.unsqueeze(0),
            "img_path" : row["img_path"],
            "sample_id": row["sample_id"],
        }

# ── Train / Val split ──────────────────────────────────────
valid_train_df = train_df[train_df["has_mask"]].reset_index(drop=True)
n_val          = max(1, int(len(valid_train_df) * CFG["val_split"]))
indices        = np.random.permutation(len(valid_train_df))
val_idx        = indices[:n_val]
train_idx      = indices[n_val:]

tr_df  = valid_train_df.iloc[train_idx].reset_index(drop=True)
vl_df  = valid_train_df.iloc[val_idx ].reset_index(drop=True)

# Save split info
split_info = pd.concat([
    tr_df.assign(split_role="train"),
    vl_df.assign(split_role="val"),
], ignore_index=True)
split_info.to_csv(f"{DIRS['preproc']}/data_splits.csv", index=False)

# ── DataLoaders ────────────────────────────────────────────
train_ds = HC18Dataset(tr_df, transform=train_transform, mode="train")
val_ds   = HC18Dataset(vl_df, transform=val_transform,  mode="val")
test_ds  = HC18Dataset(test_df, transform=val_transform, mode="test")

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=2, pin_memory=True)

print("="*60)
print("Data Split Summary")
print("="*60)
print(f"  Training samples   : {len(train_ds)}")
print(f"  Validation samples : {len(val_ds)}")
print(f"  Test samples       : {len(test_ds)}")
print(f"  Train batches/epoch: {len(train_loader)}")
print("Split info saved to:", f"{DIRS['preproc']}/data_splits.csv")